# TinyVLM v2 -- SigLIP2 + Qwen2.5-0.5BRebuild of the pipeline after diagnosing why run 1 produced one or two fixedanswers regardless of the image.## What went wrong in run 1| Root cause | Evidence | Fix ||---|---|---|| Projector alignment corpus ~350x too small | `--max_examples 8000` on `captions.txt` = ~1600 distinct images (5 captions each, consecutive); 500 optimizer steps | Flickr8k + Flickr30k, ~39k images / ~185k pairs || Stage-1 loss looked fine but wasn't | `loss = loss/accum_steps` was assigned *before* `set_postfix`, so the logged `0.786` was really ~3.14 | logging reports the true per-token loss || Trained to emit exactly one word | VQAv2 targets tokenize to `['red', '<|im_end|>']` -- 2 supervised tokens out of 238 | mix VQA + detailed captions; short answers carry an explicit prompt hint || Visual tokens far outside the embedding manifold | SigLIP token norm 47.2 vs Qwen embedding norm 0.45 (~104x) | projector ends in LayerNorm initialised to the LLM's own embedding std -> 1.01x || LoRA never touched the MLP block | `target_modules=[q,k,v,o]` only | + `gate_proj, up_proj, down_proj` || 1.1 GB adapter | adding `<image>` forced `resize_token_embeddings`, so PEFT serialised the whole embedding matrix | reuse Qwen's existing `<|image_pad|>`; adapter is ~35 MB || GPU starved | `num_workers=0`, full-resolution JPEG decode every epoch -> 1.33 it/s | prep-time resize + worker processes + fp16 AMP || No way to notice failure before upload | no held-out loss, no sample generations | eval loss + generations every `--eval_every` steps, plus an explicit grounding gate between stages |## Session setup- **Accelerator:** GPU T4 x2- **Internet:** On- **Add data:** Kaggle dataset `adityajn105/flickr8k`Budget: roughly 1.5-2 h data prep, ~2 h Stage 1, ~1.5 h Stage 2.

## 1. Environment

In [ ]:
!pip install -q "transformers>=4.45" "peft>=0.11" accelerate datasets huggingface_hub

In [ ]:
import torch, transformers, peftprint("torch", torch.__version__, "| cuda", torch.cuda.is_available())print("transformers", transformers.__version__, "| peft", peft.__version__)print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU -- fix the accelerator setting")

## 2. Get the codeThe notebook clones the repo instead of `%%writefile`-ing the modules inline.Run 1 had the model/training code duplicated in both places and they had alreadydrifted (the notebook's stage-1 script had gradient accumulation, the repo'sdid not). One source of truth.

In [ ]:
import os, sysREPO = "/kaggle/working/VLM"if not os.path.exists(REPO):    !git clone -q https://github.com/patelatwork/VLM.git {REPO}else:    !cd {REPO} && git pull -q%cd {REPO}sys.path[:0] = [f"{REPO}/Model", f"{REPO}/Training", f"{REPO}/Inference"]!ls Model Training Inference

## 3. DataFour corpora, all written to one shared image directory as JPEGs with the shortside resized to 256. Run 1 symlinked full-resolution originals and re-decodedthem every epoch.| corpus | role | rows ||---|---|---|| Flickr8k | alignment captions | ~40k (8k images) || Flickr30k | alignment captions | ~145k (29k images) || VQAv2 | short answers | 40k || LLaVA-ReCap-118K | detailed descriptions | 25k |

In [ ]:
!ls /kaggle/input

In [ ]:
# Flickr8k -- from the Kaggle input you attached in the sidebar.# If this errors, the traceback prints the directory tree it searched.!python Training/prepare_data.py --task flickr8k \    --kaggle_input_dir /kaggle/input \    --out_jsonl /kaggle/working/data/cap8k.jsonl \    --out_image_dir /kaggle/working/data/images

In [ ]:
# Flickr30k -- ~29k more distinct images. This is the single biggest change# from run 1: the projector now sees ~24x more visual variety.!python Training/prepare_data.py --task flickr30k \    --out_jsonl /kaggle/working/data/cap30k.jsonl \    --out_image_dir /kaggle/working/data/images

In [ ]:
!python Training/prepare_data.py --task vqav2 --max_examples 40000 \    --out_jsonl /kaggle/working/data/vqa.jsonl \    --out_image_dir /kaggle/working/data/images

In [ ]:
# Detailed multi-sentence descriptions -- this is what restores the model's# ability to write prose instead of a single word.!python Training/prepare_data.py --task recap --max_examples 25000 \    --out_jsonl /kaggle/working/data/recap.jsonl \    --out_image_dir /kaggle/working/data/images

In [ ]:
# Stage 1 = pure captioning. Stage 2 = a deliberate mix of answer styles.!python Training/prepare_data.py --task mix \    --inputs /kaggle/working/data/cap8k.jsonl /kaggle/working/data/cap30k.jsonl \    --out_jsonl /kaggle/working/data/stage1.jsonl!python Training/prepare_data.py --task mix \    --inputs /kaggle/working/data/vqa.jsonl /kaggle/working/data/recap.jsonl /kaggle/working/data/cap8k.jsonl \    --caps 40000 25000 20000 \    --out_jsonl /kaggle/working/data/stage2.jsonl

In [ ]:
import json, collectionsfor name in ["stage1", "stage2"]:    rows = [json.loads(l) for l in open(f"/kaggle/working/data/{name}.jsonl", encoding="utf-8")]    styles = collections.Counter(r.get("style", "caption") for r in rows)    ans_len = [len(t["value"].split()) for r in rows for t in r["conversations"] if t["from"] == "gpt"]    print(f"{name}: {len(rows):,} rows | styles {dict(styles)}")    print(f"   answer length: mean {sum(ans_len)/len(ans_len):.1f} words, "          f"min {min(ans_len)}, max {max(ans_len)}")print("\nsample stage2 rows:")for r in rows[:3]:    print(" ", json.dumps(r)[:220])

## 4. Stage 1 -- projector alignmentVision encoder and LLM stay frozen; only the projector trains. Gradients stillflow *through* the LLM to reach it, so this is not free, but no LLM weights move.Watch the sample generations printed every 500 steps. If two different imagesproduce the same caption, stop -- something is wrong and Stage 2 will notrescue it.

In [ ]:
!python Training/train.py --stage 1 \    --data /kaggle/working/data/stage1.jsonl \    --image_root /kaggle/working/data/images \    --output_dir /kaggle/working/ckpt/stage1 \    --epochs 2 --batch_size 32 --accum_steps 1 --lr 1e-3 \    --num_workers 4 --eval_every 500 --save_every 4000

### Grounding gateRun this before spending time on Stage 2. Three visually different images, onequestion. Identical answers = the projector did not learn, and nothingdownstream will fix that.

In [ ]:
import glob, jsonrows = [json.loads(l) for l in open("/kaggle/working/data/stage2.jsonl", encoding="utf-8")][:400]seen, picks = set(), []for r in rows:                      # three images from different source corpora    tag = r["image"].split("_")[0]    if tag not in seen:        seen.add(tag); picks.append("/kaggle/working/data/images/" + r["image"])    if len(picks) == 3: breakpicks += ["/kaggle/working/data/images/" + rows[i]["image"] for i in range(3) if len(picks) < 3]print(picks)!python Inference/diagnose_grounding.py \    --projector_ckpt /kaggle/working/ckpt/stage1/projector.pt \    --images {" ".join(picks)}

## 5. Stage 2 -- LoRA instruction tuningLoRA on attention **and** MLP projections; the projector keeps training at amuch lower LR so it is refined rather than overwritten.

In [ ]:
!python Training/train.py --stage 2 \    --data /kaggle/working/data/stage2.jsonl \    --image_root /kaggle/working/data/images \    --projector_ckpt /kaggle/working/ckpt/stage1/projector.pt \    --output_dir /kaggle/working/ckpt/stage2 \    --epochs 3 --batch_size 16 --accum_steps 1 \    --lr 2e-4 --projector_lr 2e-5 --lora_r 16 --lora_alpha 32 \    --num_workers 4 --eval_every 500 --save_every 4000

### Grounding gate, again -- now with the adapter

In [ ]:
!python Inference/diagnose_grounding.py \    --projector_ckpt /kaggle/working/ckpt/stage2/projector.pt \    --lora_dir /kaggle/working/ckpt/stage2/lora_adapter \    --images {" ".join(picks)}

## 6. Qualitative checkBoth answer styles on the same images. Descriptive prompts should givesentences; the short-answer hint should give a word or two. If everything comesback one word, the style mix in Stage 2 did not take.

In [ ]:
import syssys.path[:0] = ["/kaggle/working/VLM/Model", "/kaggle/working/VLM/Inference"]from inference import load_model, answerfrom PIL import Imageimport matplotlib.pyplot as pltmodel = load_model("/kaggle/working/ckpt/stage2/projector.pt",                   "/kaggle/working/ckpt/stage2/lora_adapter", "cuda")tests = [(p, "What is in this image?") for p in picks]fig, axes = plt.subplots(1, len(tests), figsize=(5 * len(tests), 5))for ax, (path, q) in zip(axes if len(tests) > 1 else [axes], tests):    img = Image.open(path).convert("RGB")    long_a = answer(model, img, q, "cuda")    short_a = answer(model, img, q, "cuda", short_answer=True)    ax.imshow(img); ax.axis("off")    ax.set_title(f"long: {long_a[:70]}\nshort: {short_a[:40]}", fontsize=8, wrap=True)    print(f"{path}\n  descriptive: {long_a}\n  short      : {short_a}\n")plt.tight_layout(); plt.show()

In [ ]:
# Held-out VQA accuracy, exact match on the short-answer prompt.import json, randomrows = [json.loads(l) for l in open("/kaggle/working/data/vqa.jsonl", encoding="utf-8")]random.Random(1).shuffle(rows)sample = rows[:150]hits = 0for r in sample:    q = r["conversations"][0]["value"].replace("<image>", "").split("\nAnswer the question")[0].strip()    gt = r["conversations"][1]["value"].strip().lower()    pred = answer(model, "/kaggle/working/data/images/" + r["image"], q, "cuda",                  short_answer=True).strip().lower().rstrip(".")    hits += (pred == gt)print(f"exact-match on {len(sample)} held-in VQA rows: {hits/len(sample):.1%}")print("(a language-prior-only model lands near 25-30%; grounded should be well above)")

## 7. Push to the Hub

In [ ]:
import ostry:    from kaggle_secrets import UserSecretsClient    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")except Exception:    print("No Kaggle secret named HF_TOKEN -- add one under Add-ons > Secrets.")from huggingface_hub import loginlogin(token=os.environ["HF_TOKEN"]) if "HF_TOKEN" in os.environ else login()

In [ ]:
!python push_to_hub.py \    --repo_id dhruvpatel93/tinyvlm-vqa \    --projector_ckpt /kaggle/working/ckpt/stage2/projector.pt \    --lora_dir /kaggle/working/ckpt/stage2/lora_adapter \    --meta /kaggle/working/ckpt/stage2/train_meta.json

## 8. SpaceThe Space needs `Model/model.py`, `Model/dataset.py`, `Inference/inference.py`and `Inference/app_gradio.py` flattened into its root, plus`Inference/requirements.txt`. Set `HF_REPO_ID` as a Space variable and`app_gradio.py` pulls the weights from the Hub itself.Do not hand-edit the prompt format in the Space -- `inference.answer` uses`dataset.build_prompt`, the same function that built the training data. Run 1rendered the chat template separately in each place.